# Genesis AI — opportunistic Colab accelerator

This notebook is an **optional** free accelerator entry point. It never replaces GitHub CI, the CPU research farm, frozen evaluation, or checkpoint-promotion gates. Colab hardware and quotas are availability-dependent.

Before running training: select a **GPU** runtime, provide an enabled accelerator job manifest that survived CPU screening, and upload the matching `cpu-farm-summary.json`. TPU detection is supported, but Genesis PyTorch/XLA training is not enabled yet.

In [ ]:
import os, pathlib, subprocess, sys
REPO = pathlib.Path('/content/genesis-ai')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/NahuelGenchi/genesis-ai.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-e', str(REPO)], check=True)
os.chdir(REPO)


In [ ]:
from genesis_ai.accelerator_job import detect_accelerator
runtime = detect_accelerator('colab')
runtime


## Optional durable checkpoints
Mount Google Drive if you want checkpoints to persist across Colab session loss. The accelerator runner resumes automatically when `latest.pt` already exists in the selected output directory.

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/genesis-ai-output'
else:
    OUTPUT_DIR = '/content/genesis-ai-output'
print('Output:', OUTPUT_DIR)


## Configure the screened experiment
`JOB_MANIFEST` must be a committed, enabled job under `accelerators/jobs/`. Upload the matching CPU farm artifact as `/content/cpu-farm-summary.json`. The runner refuses jobs that are absent from `expensive_stage_eligible`.

In [ ]:
JOB_MANIFEST = 'accelerators/jobs/replace-me.json'
CPU_SUMMARY = '/content/cpu-farm-summary.json'
RUN = False  # Change only after reviewing the manifest and CPU summary.
if RUN:
    subprocess.run([
        sys.executable, '-m', 'genesis_ai.accelerator_job', 'run',
        '--job', JOB_MANIFEST,
        '--cpu-summary', CPU_SUMMARY,
        '--platform', 'colab',
        '--output-dir', OUTPUT_DIR,
    ], check=True)
else:
    print('Dry by default. Set RUN=True only for a CPU-screened model job.')
